# Crisis Mining V11 (Colab) — batch 2

Second batch of V11 crisis mining. Same setup, different seeds.
Model: pillar2x2_epoch_10. Evaluator: feature_value_weights_2x.npz.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz`
2. `pillar2x2_epoch_10.pt`
3. `feature_value_weights_2x.npz`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar2x2_epoch_10.pt /content/alphatrain/data/model.pt
!cp {DRIVE}/feature_value_weights_2x.npz /content/alphatrain/data/feature_value_weights_2x.npz
print(f'Model: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')
print(f'Feature weights: {os.path.getsize("/content/alphatrain/data/feature_value_weights_2x.npz")} bytes')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === CRISIS MINING V11 BATCH 2 ===
# Instance 1: 800000-810000 (batch 1)
# Instance 2: 810000-820000 (this batch)
SEED_START = 810000
SEED_END = 820000
WORKERS = 20
BATCH_SIZE = 8
POLICY_MAX_TURNS = 3000   # cap policy-only probe length (strong policies survive forever otherwise)
SAVE_DIR = f'{DRIVE}/crisis_v11'
# =============================

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} probes)')

!python -m alphatrain.scripts.crisis_mining \
    --model /content/alphatrain/data/model.pt \
    --feature-value-weights /content/alphatrain/data/feature_value_weights_2x.npz \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --recovery-turns 15 --recovery-sims 600 \
    --prevention-turns 30 --prevention-sims 600 \
    --continue-turns 500 \
    --policy-max-turns {POLICY_MAX_TURNS} \
    --workers {WORKERS} --device cuda --batch-size {BATCH_SIZE} \
    --compile \
    --save-dir {SAVE_DIR}